In [123]:
import pandas as pd
import numpy as np
import json

In [124]:
#Parameters
emme_csv_path = '/content/EMME.csv'
scale_definition_json_path = '/content/canmovesScaleDefinition.json'
config_json_path = '/content/canmovesRunSpecConfig.json'
basic_fuel_csv_path = '/content/Fuel Distribution_Basic.csv'
epa_fuel_csv_path = '/content/Fuel Distribution_EPA.csv'
age_csv_path = '/content/Age.csv'
df_activity_map_total_volume_path = '/content/df_activity_map_total_volume.csv'
df_activity_map_total_speed_path = '/content/df_activity_map_total_speed.csv'
df_activity_map_total_coldstart_path = '/content/df_activity_map_total_coldstart.csv'
df_activity_map_total_vkt_path = '/content/df_activity_map_total_vkt.csv'
df_activity_map_fuel_path = '/content/df_activity_map_fuel.csv'
df_activity_map_source_path = '/content/df_activity_map_source.csv'
df_activity_map_model_path = '/content/df_activity_map_model.csv'
df_activity_chart_by_model_per_fuel_path = '/content/df_activity_chart_by_model_per_fuel.csv'
df_activity_chart_by_model_per_source_path = '/content/df_activity_chart_by_model_per_source.csv'
df_activity_chart_by_source_per_fuel_path = '/content/df_activity_chart_by_source_per_fuel.csv'
df_activity_chart_total_per_source_path = '/content/df_activity_chart_total_per_source.csv'
df_activity_chart_total_per_fuel_path = '/content/df_activity_chart_total_per_fuel.csv'
df_activity_chart_total_per_model_path = '/content/df_activity_chart_total_per_model.csv'

#Functions

In [150]:
def read_emme_file(emme_csv_path):
    raw_df = pd.read_csv(emme_csv_path)
    raw_df = raw_df.dropna()
    return raw_df

In [151]:
def get_existing_vehicle_types(emme_df):
    vehicle_types = ["LDV", "LDT", "MDV", "HDV", "Bus"]
    return [vehicle_type for vehicle_type in vehicle_types if f"{vehicle_type}_Volume" in emme_df.columns]


In [152]:
def add_total_vehicle_columns(emme_df):
    df = emme_df.copy()
    vehicle_types = get_existing_vehicle_types(df)

    volume_columns = [f"{vehicle_type}_Volume" for vehicle_type in vehicle_types]
    df["Total_Volume"] = df[volume_columns].sum(axis=1)

    speed_numerator = sum(
        df[f"{vehicle_type}_Speed_kph"] * df[f"{vehicle_type}_Volume"]
        for vehicle_type in vehicle_types
        if f"{vehicle_type}_Speed_kph" in df.columns
    )

    coldstart_numerator = sum(
        df[f"{vehicle_type}_ColdStart_pct"] * df[f"{vehicle_type}_Volume"]
        for vehicle_type in vehicle_types
        if f"{vehicle_type}_ColdStart_pct" in df.columns
    )

    df["Total_Speed_kph"] = np.where(
        df["Total_Volume"] > 0,
        speed_numerator / df["Total_Volume"],
        0,
    )

    df["Total_ColdStart_pct"] = np.where(
        df["Total_Volume"] > 0,
        coldstart_numerator / df["Total_Volume"],
        0,
    )

    return df

In [153]:
def add_vkt_columns(emme_df):
    df = emme_df.copy()
    vehicle_types = get_existing_vehicle_types(df)

    if "Total_Volume" in df.columns:
        vehicle_types = vehicle_types + ["Total"]

    for vehicle_type in vehicle_types:
        df[f"{vehicle_type}_VKT"] = df[f"{vehicle_type}_Volume"] * df["Length_km"]

    return df

In [154]:
def prepare_emme_input(emme_csv_path):
    emme_df = read_emme_file(emme_csv_path)
    emme_df = add_total_vehicle_columns(emme_df)
    emme_df = add_vkt_columns(emme_df)

    return emme_df

In [155]:
def read_fleet_classification(scale_definition_json_path):
    with open(scale_definition_json_path, "r") as file:
        scale_definition = json.load(file)

    return scale_definition["fleetClassification"]

In [156]:
def select_fuel_distribution_path(scale_definition_json_path, basic_fuel_csv_path, epa_fuel_csv_path):
    fleet_classification = read_fleet_classification(scale_definition_json_path)

    if fleet_classification == "basic5":
        return basic_fuel_csv_path

    return epa_fuel_csv_path


In [157]:
def prepare_subclass_vkt_distribution(fuel_distribution_csv_path, fleet_classification):
    fuel_df = pd.read_csv(fuel_distribution_csv_path)

    fuel_df = fuel_df[["Class #", "Sub-class #", "Percent of Class"]].copy()

    fuel_df["Class #"] = pd.to_numeric(fuel_df["Class #"], errors="coerce").astype(int)
    fuel_df["Sub-class #"] = pd.to_numeric(fuel_df["Sub-class #"], errors="coerce").astype(int)
    fuel_df["Percent of Class"] = pd.to_numeric(fuel_df["Percent of Class"], errors="coerce")

    if fleet_classification == "basic5":
        class_mapping = {
            1: "LDV",
            2: "LDT",
            3: "MDV",
            4: "HDV",
            5: "Bus",
        }
    else:
        class_mapping = {
            1: "LDV",
            2: "LDT",
            3: "HDV",
            4: "Bus",
        }

    source_type_names = [
        "LDV-mini",
        "LDV-Eco",
        "LDV-Large",
        "LDT1",
        "LDT2",
        "LDT3",
        "LDT4",
        "HDV2b3",
        "HDV3",
        "HDV4",
        "HDV5",
        "HDV6",
        "HDV7",
        "HDV8a",
        "HDV8b",
        "Bus - SS",
        "Bus - SL",
        "Bus - TO",
        "Bus - TN",
        "Bus - TL",
        "Bus - TS",
    ]

    fuel_df["vehicle_type"] = fuel_df["Class #"].map(class_mapping)
    fuel_df["sourceTypeID"] = source_type_names[:len(fuel_df)]

    class_total = fuel_df.groupby("Class #")["Percent of Class"].transform("sum")
    fuel_df["subclass_fraction"] = fuel_df["Percent of Class"] / class_total

    return fuel_df[["vehicle_type", "sourceTypeID", "Class #", "Sub-class #", "subclass_fraction"]]

In [158]:
def breakdown_vkt_to_subclass_columns(emme_df, subclass_distribution_df):
    df = emme_df.copy()

    for _, row in subclass_distribution_df.iterrows():
        vehicle_type = row["vehicle_type"]
        source_type = row["sourceTypeID"]
        subclass_fraction = row["subclass_fraction"]

        vkt_column = f"{vehicle_type}_VKT"
        output_column = f"{source_type}_VKT"

        if vkt_column in df.columns:
            df[output_column] = df[vkt_column] * subclass_fraction

    return df

In [159]:
def extract_total_emme_dataframes(emme_subclass_vkt_df):
    df = emme_subclass_vkt_df.copy()

    df["inode"] = df["inode"].astype(int)
    df["jnode"] = df["jnode"].astype(int)

    df["linkID"] = df["inode"].astype(str) + "-" + df["jnode"].astype(str)

    total_volume_df = df[["linkID", "Total_Volume"]].copy()

    total_speed_df = df[["linkID", "Total_Speed_kph"]].copy()

    total_coldstart_df = df[["linkID", "Total_ColdStart_pct"]].copy()

    total_vkt_df = df[["linkID", "Total_VKT"]].copy()

    return total_volume_df, total_speed_df, total_coldstart_df, total_vkt_df

In [160]:
def extract_subclass_vkt_dataframe(emme_subclass_vkt_df):
    df = emme_subclass_vkt_df.copy()
    df["inode"] = df["inode"].astype(int)
    df["jnode"] = df["jnode"].astype(int)
    df["linkID"] = df["inode"].astype(str) + "-" + df["jnode"].astype(str)

    start_column = "LDV-mini_VKT"
    subclass_vkt_columns = df.columns[df.columns.get_loc(start_column):].tolist()

    subclass_vkt_df = df[["linkID"] + subclass_vkt_columns[:-1]].copy()

    return subclass_vkt_df

In [161]:
def clean_header(name):
    name = str(name).strip()
    name = " ".join(name.split())
    name = name.replace("Tailpipe Nox", "Tailpipe NOx")
    return name

In [162]:
def prepare_fuel_distribution(selected_fuel_csv_path):
    fuel_df = pd.read_csv(selected_fuel_csv_path)
    fuel_df.columns = [clean_header(col) for col in fuel_df.columns]

    fuel_columns = {
        "Percent Gasoline": "Gasoline",
        "Percent Diesel": "Diesel",
        "Percent Nat. Gas": "Nat. Gas",
        "Percent Propane": "Propane",
        "Percent Methanol": "Methanol",
        "percent Ethanol": "Ethanol",
    }

    fuel_column_names = list(fuel_columns.keys())

    fuel_df = fuel_df[
        ["Subclass", "Class #", "Sub-class #"] + fuel_column_names
    ].copy()

    fuel_sum = fuel_df[fuel_column_names].sum(axis=1)

    zero_distribution_rows = fuel_sum == 0

    for index, row in fuel_df[zero_distribution_rows].iterrows():
        class_number = row["Class #"]

        base_distribution = fuel_df[
            (fuel_df["Class #"] == class_number) &
            (fuel_df["Sub-class #"] == 1)
        ]

        if not base_distribution.empty:
            fuel_df.loc[index, fuel_column_names] = base_distribution.iloc[0][fuel_column_names].values

    fuel_sum = fuel_df[fuel_column_names].sum(axis=1)

    fuel_df[fuel_column_names] = (
        fuel_df[fuel_column_names]
        .div(fuel_sum.replace(0, np.nan), axis=0)
        .fillna(0)
    )

    fuel_df["Subclass"] = fuel_df["Subclass"].replace(
        {
            "LDV-Econ": "LDV-Eco",
            "HDV2b": "HDV2b3",
            "Short School": "Bus - SS",
            "Long School": "Bus - SL",
            "Old Transit": "Bus - TO",
            "New Transit": "Bus - TN",
            "Long transit": "Bus - TL",
            "Short Transit": "Bus - TS",
        }
    )

    fuel_long_df = fuel_df.melt(
        id_vars="Subclass",
        value_vars=fuel_column_names,
        var_name="fuel_column",
        value_name="fuel_fraction",
    )

    fuel_long_df["fuel_type"] = fuel_long_df["fuel_column"].replace(fuel_columns)

    fuel_long_df = fuel_long_df.rename(columns={"Subclass": "vehicle_type"})

    return fuel_long_df[["vehicle_type", "fuel_type", "fuel_fraction"]]

In [163]:
def expand_subclass_vkt_by_fuel(subclass_vkt_df, fuel_distribution_df):
    vkt_columns = [col for col in subclass_vkt_df.columns if col.endswith("_VKT")]

    long_vkt_df = subclass_vkt_df.melt(
        id_vars="linkID",
        value_vars=vkt_columns,
        var_name="sourceTypeID",
        value_name="VKT",
    )

    long_vkt_df["sourceTypeID"] = long_vkt_df["sourceTypeID"].str.replace("_VKT", "", regex=False)

    expanded_df = long_vkt_df.merge(
        fuel_distribution_df,
        left_on="sourceTypeID",
        right_on="vehicle_type",
        how="left",
    )

    expanded_df["Activity"] = expanded_df["VKT"] * expanded_df["fuel_fraction"]

    return expanded_df[
        [
            "linkID",
            "sourceTypeID",
            "fuel_type",
            "Activity",
        ]
    ].rename(columns={"fuel_type": "fuelTypeID"})

In [164]:
def summarize_subclass_fuel_vkt_by_fuel(subclass_fuel_vkt_df):
    summarized_df = (
        subclass_fuel_vkt_df
        .groupby(["linkID", "fuelTypeID"], as_index=False)["Activity"]
        .sum()
    )

    return summarized_df[["linkID", "fuelTypeID", "Activity"]]

In [165]:
def summarize_subclass_fuel_vkt_by_source(subclass_fuel_vkt_df):
    summarized_df = (
        subclass_fuel_vkt_df
        .groupby(["linkID", "sourceTypeID"], as_index=False)["Activity"]
        .sum()
    )

    return summarized_df[["linkID", "sourceTypeID", "Activity"]]

In [166]:
def get_vehicle_types():
    return [
        ("LDV-mini", "LDV-Mini", "LDV-mini"),
        ("LDV-Eco", "LDV-Economy", "LDV-Eco"),
        ("LDV-Large", "LDV-Large", "LDV-Large"),
        ("LDT1", "LDT1", "LDT1"),
        ("LDT2", "LDT2", "LDT2"),
        ("LDT3", "LDT3", "LDT3"),
        ("LDT4", "LDT4", "LDT4"),
        ("HDV2b3", "HDV2b", "HDV2b3"),
        ("HDV3", "HDV3", "HDV3"),
        ("HDV4", "HDV4", "HDV4"),
        ("HDV5", "HDV5", "HDV5"),
        ("HDV6", "HDV6", "HDV6"),
        ("HDV7", "HDV7", "HDV7"),
        ("HDV8a", "HDV8a", "HDV8a"),
        ("HDV8b", "HDV8b", "HDV8b"),
        ("Bus - SS", "Bus - SS", "Short School"),
        ("Bus - SL", "Bus - SL", "Blong Schoolus"),
        ("Bus - TO", "Bus - TO", "Old Transit"),
        ("Bus - TN", "Bus - TN", "New Tranist"),
        ("Bus - TL", "Bus - TL", "Long Transit"),
        ("Bus - TS", "Bus - TS", "Short Transit"),
    ]

In [167]:
def prepare_age_distribution(age_csv_path):
    age_df = pd.read_csv(age_csv_path)
    age_df.columns = [clean_header(col) for col in age_df.columns]

    vehicle_types = get_vehicle_types()
    age_rename = {clean_header(age_header): vehicle_type for vehicle_type, raw_prefix, age_header in vehicle_types}

    age_df = age_df.rename(columns=age_rename)
    age_df = age_df.rename(columns={"Vehicle Age": "vehicle_age"})

    vehicle_columns = [vehicle_type for vehicle_type, raw_prefix, age_header in vehicle_types]
    age_df = age_df[["vehicle_age"] + vehicle_columns]

    long_df = age_df.melt(id_vars="vehicle_age", value_vars=vehicle_columns, var_name="vehicle_type", value_name="age_fraction")

    long_df["vehicle_age"] = pd.to_numeric(long_df["vehicle_age"], errors="coerce")
    long_df["age_fraction"] = pd.to_numeric(long_df["age_fraction"], errors="coerce")

    long_df["age_fraction"] = long_df.groupby("vehicle_type")["age_fraction"].transform(lambda x: x / x.sum())

    return long_df[["vehicle_type", "vehicle_age", "age_fraction"]]

In [168]:
def read_selected_year(config_json_path):
    with open(config_json_path, "r") as file:
        config = json.load(file)

    return int(config["domainInfo"]["selectedYear"])

In [169]:
def expand_subclass_vkt_by_age(subclass_vkt_df, age_long_df, config_json_path):
    df = subclass_vkt_df.copy()
    age_df = age_long_df.copy()

    selected_year = read_selected_year(config_json_path)

    age_df["modelYearID"] = selected_year - age_df["vehicle_age"]
    age_df["modelYearID"] = age_df["modelYearID"].astype(int)

    vkt_columns = [col for col in df.columns if col.endswith("_VKT") and col != "Total_VKT"]

    long_vkt_df = df.melt(
        id_vars="linkID",
        value_vars=vkt_columns,
        var_name="sourceTypeID",
        value_name="VKT",
    )

    long_vkt_df["sourceTypeID"] = long_vkt_df["sourceTypeID"].str.replace("_VKT", "", regex=False)

    expanded_df = long_vkt_df.merge(
        age_df,
        left_on="sourceTypeID",
        right_on="vehicle_type",
        how="left",
    )

    expanded_df["Activity"] = expanded_df["VKT"] * expanded_df["age_fraction"]

    return expanded_df[
        [
            "linkID",
            "sourceTypeID",
            "modelYearID",
            "Activity",
        ]
    ]

In [170]:
def summarize_subclass_age_vkt_by_model_year(subclass_age_vkt_df):
    df = subclass_age_vkt_df.copy()
    summarized_df = (df.groupby(["linkID", "modelYearID"], as_index=False)["Activity"].sum())

    return summarized_df[["linkID", "modelYearID", "Activity"]]

In [171]:
def expand_activity_map_source_by_fuel(df_activity_map_source, fuel_distribution_df):
    source_df = df_activity_map_source.copy()
    fuel_df = fuel_distribution_df.copy()

    source_df = (source_df.groupby(["sourceTypeID"], as_index=False)["Activity"].sum())

    expanded_df = source_df.merge(fuel_df, left_on="sourceTypeID", right_on="vehicle_type", how="left")

    expanded_df["Activity"] = expanded_df["Activity"] * expanded_df["fuel_fraction"]

    expanded_df = expanded_df.rename(
        columns={
            "fuel_type": "fuelTypeID",
        }
    )

    pivot_df = expanded_df.pivot_table(index="sourceTypeID", columns="fuelTypeID", values="Activity", aggfunc="sum", fill_value=0).reset_index()
    pivot_df.columns.name = None
    return pivot_df

In [172]:
def summarize_age_vkt_by_source_and_model_year(subclass_age_vkt_df):
    df = subclass_age_vkt_df.copy()

    summary_df = (df.groupby(["modelYearID", "sourceTypeID"], as_index=False)["Activity"].sum())

    pivot_df = summary_df.pivot_table(
        index="modelYearID",
        columns="sourceTypeID",
        values="Activity",
        aggfunc="sum",
        fill_value=0
    ).reset_index()

    pivot_df.columns.name = None

    return pivot_df

In [173]:
def summarize_fuel_vkt_by_source_and_fuel(subclass_fuel_vkt_df):
    df = subclass_fuel_vkt_df.copy()

    summary_df = (df.groupby(["sourceTypeID", "fuelTypeID"], as_index=False)["Activity"].sum())

    pivot_df = summary_df.pivot_table(
        index="sourceTypeID",
        columns="fuelTypeID",
        values="Activity",
        aggfunc="sum",
        fill_value=0
    ).reset_index()

    pivot_df.columns.name = None

    return pivot_df

In [174]:
def generate_activity_summary_dataframes(subclass_age_vkt_df, subclass_fuel_vkt_df):
    age_df = subclass_age_vkt_df.copy()
    fuel_df = subclass_fuel_vkt_df.copy()

    source_summary_df = (
        age_df
        .groupby("sourceTypeID", as_index=False)["Activity"]
        .sum()
    )

    model_year_summary_df = (
        age_df
        .groupby("modelYearID", as_index=False)["Activity"]
        .sum()
    )

    fuel_summary_df = (
        fuel_df
        .groupby("fuelTypeID", as_index=False)["Activity"]
        .sum()
    )

    return source_summary_df, model_year_summary_df, fuel_summary_df

#Main Code

In [175]:
if __name__ == "__main__":
  fleet_classification = read_fleet_classification(scale_definition_json_path)

  selected_fuel_csv_path = select_fuel_distribution_path(scale_definition_json_path, basic_fuel_csv_path, epa_fuel_csv_path)

  subclass_distribution_df = prepare_subclass_vkt_distribution(selected_fuel_csv_path, fleet_classification)

  emme_df = prepare_emme_input(emme_csv_path)

  emme_subclass_vkt_df = breakdown_vkt_to_subclass_columns(emme_df, subclass_distribution_df)

  df_activity_map_total_volume, df_activity_map_total_speed, df_activity_map_total_coldstart, df_activity_map_total_vkt = extract_total_emme_dataframes(emme_subclass_vkt_df)

  subclass_vkt_df = extract_subclass_vkt_dataframe(emme_subclass_vkt_df)

  fuel_distribution_df = prepare_fuel_distribution(selected_fuel_csv_path)

  subclass_fuel_vkt_df = expand_subclass_vkt_by_fuel(subclass_vkt_df, fuel_distribution_df)

  df_activity_map_fuel = summarize_subclass_fuel_vkt_by_fuel(subclass_fuel_vkt_df)

  df_activity_map_source = summarize_subclass_fuel_vkt_by_source(subclass_fuel_vkt_df)

  age_long_df = prepare_age_distribution(age_csv_path)

  subclass_age_vkt_df = expand_subclass_vkt_by_age(subclass_vkt_df, age_long_df, config_json_path)

  df_activity_map_model = summarize_subclass_age_vkt_by_model_year(subclass_age_vkt_df)

  df_activity_chart_by_model_per_fuel = expand_activity_map_source_by_fuel(df_activity_map_source, fuel_distribution_df)

  df_activity_chart_by_model_per_source = summarize_age_vkt_by_source_and_model_year(subclass_age_vkt_df)

  df_activity_chart_by_source_per_fuel = summarize_fuel_vkt_by_source_and_fuel(subclass_fuel_vkt_df)

  df_activity_chart_total_per_source, df_activity_chart_total_per_model, df_activity_chart_total_per_fuel = generate_activity_summary_dataframes(subclass_age_vkt_df, subclass_fuel_vkt_df)

#Print

In [122]:
df_activity_map_total_volume.to_csv(df_activity_map_total_volume_path, index=False)
df_activity_map_total_speed.to_csv(df_activity_map_total_speed_path, index=False)
df_activity_map_total_coldstart.to_csv(df_activity_map_total_coldstart_path, index=False)
df_activity_map_total_vkt.to_csv(df_activity_map_total_vkt_path, index=False)
df_activity_map_fuel.to_csv(df_activity_map_fuel_path, index=False)
df_activity_map_source.to_csv(df_activity_map_source_path, index=False)
df_activity_map_model.to_csv(df_activity_map_model_path, index=False)
df_activity_chart_by_model_per_fuel.to_csv(df_activity_chart_by_model_per_fuel_path, index=False)
df_activity_chart_by_model_per_source.to_csv(df_activity_chart_by_model_per_source_path, index=False)
df_activity_chart_by_source_per_fuel.to_csv(df_activity_chart_by_source_per_fuel_path, index=False)
df_activity_chart_total_per_source.to_csv(df_activity_chart_total_per_source_path, index=False)
df_activity_chart_total_per_model.to_csv(df_activity_chart_total_per_model_path, index=False)
df_activity_chart_total_per_fuel.to_csv(df_activity_chart_total_per_fuel_path, index=False)